In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Euler vehicles resold **June 2025** — SoH at resale (real BMS-capacity SoH)

Unlike the Mahindra resold group (feed-only, no SoH possible — see `mahindra/examine_distance_per_soc`),
Euler 2023+ vehicles carry remaining-capacity, so we compute the **validated BMS-capacity SoH** and read
off the **measured residual SoH at the resale date** (a real input for resale pricing / warranty-residual).

**Data reality (honest):** of 8 Euler vehicles resold June 2025, only **3 have usable SoH** — the other
**5 report zero/broken `batteryRemainingCapacity`** (BMS gap). The 3 are **Hiload cargo models with
larger packs (~150–176 Ah** vs the ~133 Ah passenger cohort), so `euler_features` now uses an
**adaptive per-vehicle pack window**. They are also **short-history, repossessed** vehicles (6–11 months),
so SoH-at-resale is the **measured** value clamped to the data span, and the dashed forecast beyond the
last reading is for context only. Markers: registration ▼, resale ★, 80% EoFL, 5-yr warranty.

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots
import euler_features, euler_model as em, config

euler_features.main()   # rebuild feature table to include the newly-imported resold vehicles
meu = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
emodel = em.train(em.build_transitions(meu))

rs = pd.read_csv('data/Repo - Pricing - Sold.csv')
rs['reg'] = pd.to_datetime(rs['Registration Date'], format='%d-%b-%Y', errors='coerce')
rs['resale'] = pd.to_datetime(rs['Resale Date'], format='%d-%b-%Y', errors='coerce')
rs['odo'] = pd.to_numeric(rs['Odometer reading (current)'], errors='coerce')
eu_jun = rs[(rs['OEM'].astype(str).str.contains('euler', case=False, na=False)) &
            (rs['resale'].dt.year == 2025) & (rs['resale'].dt.month == 6)]
have = set(meu['vin'].unique())
n_total = len(eu_jun)
rows = []
for _, r in eu_jun.iterrows():
    vin = r['VIN']
    if vin not in have:
        continue
    g = meu[meu['vin'] == vin].sort_values('month'); reg_t = r['reg']
    last_age = float(g['age_months'].iloc[-1]); H = max(int(round(60 - last_age)), 1)
    fc = np.array(em.free_run(g, emodel, H))
    a_obs = ((g['month'] - reg_t).dt.days / 365.25).to_numpy(); s_obs = g['soh'].to_numpy()
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - reg_t).days / 365.25 for k in range(1, H + 1)])
    resale_age = (r['resale'] - reg_t).days / 365.25
    # SoH at resale = MEASURED health, interpolated on OBSERVED data and clamped to the data span —
    # NOT a wild forecast extrapolation (these are short-history repo'd vehicles).
    soh_resale = float(np.interp(resale_age, a_obs, s_obs))
    extrap = bool(resale_age > a_obs[-1] + 0.05)          # resale falls after the last reading
    rows.append(dict(vin=vin, model=r['Model'], reg=reg_t, resale=r['resale'], odo=r['odo'], n_months=len(g),
                     a_obs=a_obs, s_obs=s_obs, a_fc=a_fc, s_fc=fc, resale_age=resale_age,
                     soh_resale=soh_resale, extrap=extrap, soh_now=float(s_obs[-1]), wyr=5))
ds = sorted(rows, key=lambda d: d['soh_resale'])
print(f'{len(ds)} of {n_total} Euler vehicles resold June 2025 have BMS-capacity SoH '
      f'({n_total - len(ds)} skipped: zero/broken remaining-capacity reading):', [d['vin'][-6:] for d in ds])

## SoH trajectory + resale point (per vehicle)

In [ ]:
n = len(ds); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
titles = [f"{d['vin'][-6:]} {str(d['model'])[:10]}  resale SoH {d['soh_resale']:.0f}%  ({d['odo']:.0f}km)" for d in ds]
fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=titles, vertical_spacing=0.13, horizontal_spacing=0.06)
for i, d in enumerate(ds):
    r, c = i // ncol + 1, i % ncol + 1
    risk = float(np.interp(d['wyr'], np.concatenate([d['a_obs'], d['a_fc']]), np.concatenate([d['s_obs'], d['s_fc']]))) < 78
    fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', row=r, col=c,
                    line=dict(color='royalblue', width=1, dash='dot'), showlegend=False, hoverinfo='skip')
    fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', row=r, col=c, name='actual BMS SoH',
                    line=dict(color='royalblue', width=1.5), marker=dict(size=4), legendgroup='a', showlegend=(i == 0))
    fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', row=r, col=c, name='forecast',
                    line=dict(color=('crimson' if risk else 'seagreen'), width=2.2, dash='dash'), legendgroup='f', showlegend=(i == 0))
    fig.add_scatter(x=[d['resale_age']], y=[d['soh_resale']], mode='markers', row=r, col=c, name='resale',
                    marker=dict(symbol='star', size=12, color='darkorange'), legendgroup='r', showlegend=(i == 0))
    fig.add_vline(x=d['resale_age'], line=dict(color='darkorange', width=1.2, dash='dot'), row=r, col=c)
    fig.add_hline(y=80, line=dict(color='red', width=1, dash='dot'), row=r, col=c)
    fig.add_vline(x=d['wyr'], line=dict(color='green', width=1.1, dash='dashdot'), row=r, col=c)
    fig.update_xaxes(title_text='age (yr)', range=[0, 5.3], row=r, col=c, title_font_size=10)
    fig.update_yaxes(range=[58, 101], row=r, col=c)
fig.update_layout(height=300 * nrow, width=1150, template='plotly_white',
                  title_text='Euler resold June 2025 — BMS-capacity SoH, ★ = SoH at resale', legend=dict(orientation='h', y=1.04, x=0))
fig.show()

## Residual-SoH-at-resale summary

In [ ]:
tab = pd.DataFrame([{'vin': d['vin'], 'model': d['model'], 'registered': d['reg'].date(),
                     'resale': d['resale'].date(), 'months_data': d['n_months'],
                     'odometer_km': int(d['odo']) if pd.notna(d['odo']) else None,
                     'SoH_at_resale_%': round(d['soh_resale'], 1), 'SoH_now_%': round(d['soh_now'], 1),
                     'resale_in_data': 'no (clamped)' if d['extrap'] else 'yes',
                     'above_80_EoFL': 'yes' if d['soh_resale'] >= 80 else 'NO'} for d in ds])
print(tab.to_string(index=False))
print(f"\nmean residual SoH at resale: {tab['SoH_at_resale_%'].mean():.1f}%  |  "
      f"below 80% EoFL at resale: {(tab['SoH_at_resale_%'] < 80).sum()} of {len(tab)}")
print("Note: these are short-history, repossessed vehicles. SoH at resale is the MEASURED BMS-capacity "
      "value (clamped to the data span); the dashed forecast beyond the last reading is unreliable for "
      "the 6-7 month vehicles and shown only for context.")